# L04 · Dissecting original GKD

## Goal

**Estimated time:** 45 min · **Path:** fast, full

- separate GKD lambda from beta
- implement generalized JSD
- trace gradient paths

### Current position: L03 → **L04** → L05

```text
Prompt/Data -> state source -> ... -> L04 -> ... -> fair evaluation
```

Alt text: The course map highlights L04 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L04"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L04', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

Original GKD has two axes. Lambda controls the fraction of student-generated states; beta selects the divergence at those states. They are not one hyperparameter.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

GKD has two independent axes. `lambda` mixes fixed and student-generated states; `beta` shapes generalized JSD at those states. Under this course's convention, `beta=0` is forward KL and `beta=1` is reverse KL.

Original GKD compares the **full vocabulary distribution** on a selected prefix. Modern sampled OPD can build a score-function estimator from only student-sampled tokens. Both may use on-policy states, but their estimators and variance differ.

### Production implementation: why this design

Collection chooses the state source with lambda; loss chooses divergence with beta. Keeping them in separate functions makes ablations auditable. The teacher scores under `eval()` and `no_grad()` while only the student updates.

Production code: [`core.py`](../../src/opd_study/training/core.py), [`losses.py`](../../src/opd_study/algorithms/losses.py).

In [2]:
import inspect
from opd_study.algorithms import generalized_kd_loss

objects_to_show = (generalized_kd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.losses.generalized_kd_loss
def generalized_kd_loss(
    student_logits: Tensor,
    trajectories: TrajectoryBatch,
    signals: TeacherSignals,
    *,
    beta: float = 0.5,
    temperature: float = 1.0,
) -> LossOutput:
    """Generalized JSD objective from GKD on whatever states were collected.

    State-source mixing (the GKD lambda) belongs in collection, not in this loss.
    Keeping those mechanisms separate makes on-policy/off-policy ablations auditable.
    """

    shifted_student, _, _, prediction_mask = shifted_causal_tensors(
        student_logits,
        trajectories.token_ids,
        trajectories.attention_mask,
        trajectories.response_mask,
    )
    shifted_teacher = _teacher_shifted_logits(signals, trajectories, student_logits)
    shifted_loss = torch.mul(
        generalized_jsd_from_logits(
            shifted_teacher,
            shifted_student,
            beta=beta,
            temperature=temperature,
        ),
        tempera

### Alternatives and trade-offs

Lambda may be fixed, warmed up, or adapted to performance; beta can select FKL, RKL, or intermediate JSD. Defaults first preserve paper semantics. Adaptive policies remain separate experimental variables.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L04's output? Write one sentence, then run.

In [3]:
from opd_study.math import generalized_jsd_from_logits

teacher = torch.randn(2, 5, 7)
student = torch.randn(2, 5, 7, requires_grad=True)
for beta in (0.0, 0.5, 1.0):
    value = generalized_jsd_from_logits(teacher, student, beta=beta).mean()
    print(f"beta={beta}: {value.item():.5f}")
print("beta=0 is forward KL; beta=1 is reverse KL in this repository's convention.")

beta=0.0: 0.60619
beta=0.5: 0.13255
beta=1.0: 0.60986
beta=0 is forward KL; beta=1 is reverse KL in this repository's convention.


In [4]:
generator = torch.Generator().manual_seed(42)
lambda_on_policy = 0.5
state_sources = ["student" if torch.rand((), generator=generator) < lambda_on_policy
                 else "fixed" for _ in range(8)]
print("GKD state sources:", state_sources)
print("lambda chooses states; beta chooses the divergence. They are different knobs.")

GKD state sources: ['fixed', 'fixed', 'student', 'fixed', 'student', 'fixed', 'student', 'fixed']
lambda chooses states; beta chooses the divergence. They are different knobs.


## Checks

In [5]:
forward = generalized_jsd_from_logits(teacher, student, beta=0.0)
reverse = generalized_jsd_from_logits(teacher, student, beta=1.0)
assert forward.shape == reverse.shape == (2, 5)
assert set(state_sources) == {"fixed", "student"}
print("check passed: [B,T,V] -> [B,T], with separate lambda and beta")

check passed: [B,T,V] -> [B,T], with separate lambda and beta


**Exercise (8 min):** with a fixed seed, compare state-source traces at lambda 0, .5, and 1 while keeping beta fixed. Record why the knobs are independent.

<details><summary>Check</summary>Lambda 0 selects fixed states and 1 student states; the beta/JSD formula does not change.</details>

## My recurring mistakes

### M1 — Conflating lambda and beta

- Wrong: raising lambda makes the objective reverse KL.
- Why: lambda mixes states; beta selects divergence.
- Fix: log collection and loss configs separately.
- Related check: `test_gjsd_boundaries_have_named_kl_direction`

### M2 — Equating GKD with sampled policy-gradient OPD

- Wrong: on-policy prefixes imply identical estimators.
- Why: full-vocabulary and sampled-token estimators differ in variance/memory.
- Fix: classify state source and estimator on two axes.
- Related check: `test_rollout_snapshots_are_detached_and_mode_is_restored`

## 60-second summary

1. separate GKD lambda from beta
2. implement generalized JSD
3. trace gradient paths

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`trl_gkd_reference`](https://github.com/huggingface/trl) · `1e3ba4e80dfd8c64f11022a7ae47de6a58255ca5` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)